**<h1>Classification Model - V3**
#### Classification Model V3 =  
<i>Any question regarding the notebook, please contact Robert Ford<br>
    Last Updated: 06/24/2025</i>
* AHJ Data: City and County level data sourced from Government Compensation California (GCC)
* Used to create filtered outputs from test/train data
* Started development on 06/24/2025
* Finished Iteration on -06/24/2025-

In [1]:
# Run pip updates and installs here
%pip install -U sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/10.5 MB ? eta -:--:--
   ------------------------- -------------- 6.6/10.5 MB 36.6 MB/s eta 0:00:01
   ---------------------------------------- 10.5/10.5 MB 36.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 35.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/216.1 MB ? eta -:--:--
   -- ------------------------------------- 13.4/216.1 MB 69.7 MB/s eta 0:00:03
   ----- ---------------------------------- 28.8/216.1 MB 70.2 MB/s eta 0:00:03
   ------- -------------------------------- 41.9/216.1 MB 68.3 MB/s eta 0:00:03
   ---------- ----------------------------- 56.4/216.1 MB 69.0 MB/s eta 0:00:03
   ------------ --------------------------- 65.3/216.1 MB 64.0 MB/s eta 0:00:03
   ------------- -------------------------- 71.3/216.1 MB 56.8 MB/s eta 0:00:0

In [ ]:
# Import libraries
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
import nltk
from nltk.corpus import stopwords

# Download NLTK stopwords
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Rford\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
# Keywords to KEEP
keep_dept_keywords = [
    "Planning", "Community Development", "Building Department", "Code Compliance",
    "Public Works", "Streets", "Engineering", "Construction", "Zoning"
]

keep_position_keywords = [
    "Construction", "Building", "HVAC", "Plans Inspector", "Zoning Investigator",
    "Architect", "Permit Technician", "Safety Inspector", "Planner", "Code Enforcer",
    "Plan Check Coordinator", "Licensing Specialist", "Community Development Technician"
]

# Keywords to FILTER OUT
filter_dept_keywords = [
    "Administration", "Communications", "Animal", "Finance", "Community Services",
    "Municipal Power", "Base Reuse", "City Attorney", "City Clerk", "Council", "Fire",
    "Human Resources", "Library", "IT", "Police", "Recreation", "Government Services",
    "Facilities", "Water", "Power", "Airport", "Health", "Zero Waste", "Sewer",
    "Equipment", "Transfer Station", "Janitor", "Landscaping", "Utilities", "Business",
    "Transit", "Nutrition", "Aging", "Adult", "Youth", "Homeless", "Athletics", "Law",
    "Attorney", "Management"
]

filter_position_keywords = [
    "Fitness instructor", "Recreational Leader", "Librarian", "Admin", "Clerk",
    "Secretary", "Parking Officer", "Police", "Firefighter", "Treasurer", "Employment Worker",
    "Lifeguard", "Animal", "Recycling", "Sewer", "Finance", "HR", "IT", "Parks & Rec",
    "Health", "EMT", "Custodian", "Equipment Mechanic", "Aquatics", "Traffic",
    "Community Service", "Veterinary", "Senior Citizen", "Pool", "Utilities", "Audio",
    "Communications", "Mayor", "Attendant", "Vocational Worker", "Messenger Clerk",
    "Customer Service", "Truck Operator", "Sanitation", "Painter", "Eltl Engr Assoc",
    "Laborer", "Program Assistant"
]


In [3]:
# Helper function to match keywords
def keyword_match(text, keywords):
    if pd.isna(text):
        return False
    return any(re.search(rf'\b{re.escape(k)}\b', str(text), re.IGNORECASE) for k in keywords)

# Label rows for training
def label_row(row):
    keep_dept = keyword_match(row['DepartmentOrSubdivision'], keep_dept_keywords)
    keep_pos = keyword_match(row['Position'], keep_position_keywords)
    filter_dept = keyword_match(row['DepartmentOrSubdivision'], filter_dept_keywords)
    filter_pos = keyword_match(row['Position'], filter_position_keywords)
    return int((keep_dept or keep_pos) and not (filter_dept or filter_pos))


In [4]:
# Load training data
train_df = pd.read_excel("city_training_data.xlsx")

# Apply labeling
train_df["label"] = train_df.apply(label_row, axis=1)

# Combine text for model training
train_df["combined_text"] = train_df["DepartmentOrSubdivision"].fillna('') + " " + train_df["Position"].fillna('')

# Show label distribution
train_df["label"].value_counts()


label
0    91664
1    11646
Name: count, dtype: int64

In [6]:
# Feature and Target Preparation

X = train_df["combined_text"]
y = train_df["label"]


In [ ]:
# Generate Sentence Embeddings

embedder = SentenceTransformer("all-MiniLM-L6-v2")

X_train = embedder.encode(train_df["combined_text"].tolist(), show_progress_bar=True)
y_train = train_df["label"]


✅ Model training complete.


In [ ]:
# Train the Classifier

clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

print("✅ Model trained successfully.")


In [ ]:
# Load test data
test_df = pd.read_excel("city_test_data.xlsx")

# Combine text columns
test_df["combined_text"] = test_df["DepartmentOrSubdivision"].fillna('') + " " + test_df["Position"].fillna('')

# Predict on Test Data
X_test = embedder.encode(test_df["combined_text"].tolist(), show_progress_bar=True)
test_df["predicted_label"] = clf.predict(X_test)

# Filter the results
filtered_test_df = test_df[test_df["predicted_label"] == 1]

# Display summary
print(f"Filtered rows: {len(filtered_test_df)} / {len(test_df)}")
filtered_test_df.head()


Filtered rows: 27553 / 241056


,Year,EmployerType,EmployerName,DepartmentOrSubdivision,Position,ElectedOfficial,Judicial,OtherPositions,MinPositionSalary,MaxPositionSalary,...,PensionFormula,EmployerURL,EmployerPopulation,LastUpdatedDate,EmployerCounty,SpecialDistrictActivities,IncludesUnfundedLiability,SpecialDistrictType,combined_text,predicted_label
20,2023,City,Adelanto,Streets,Maint Worker I,False,False,NaN,44363.0,49931.0,...,2%@62,https://www.ci.adelanto.ca.us/198/Human-Resources,36131,2024-06-25,San Bernardino,NaN,False,NaN,Streets Maint Worker I,1
21,2023,City,Adelanto,Streets,Maint Worker III,False,False,NaN,58955.0,66354.0,...,2%@60,https://www.ci.adelanto.ca.us/198/Human-Resources,36131,2024-06-25,San Bernardino,NaN,False,NaN,Streets Maint Worker III,1
22,2023,City,Adelanto,Streets,Maint Worker III,False,False,NaN,58955.0,66354.0,...,2%@60,https://www.ci.adelanto.ca.us/198/Human-Resources,36131,2024-06-25,San Bernardino,NaN,False,NaN,Streets Maint Worker III,1
23,2023,City,Adelanto,Streets,Maint Worker III,False,False,NaN,58955.0,66354.0,...,2%@60,https://www.ci.adelanto.ca.us/198/Human-Resources,36131,2024-06-25,San Bernardino,NaN,False,NaN,Streets Maint Worker III,1
38,2023,City,Agoura Hills,Community Development,Associate Planner,False,False,NaN,98041.0,119454.0,...,2%@55,https://www.agourahillscity.org/department/hum...,19841,2024-06-25,Los Angeles,NaN,False,NaN,Community Development Associate Planner,1


In [9]:
# Save the filtered test data
filtered_test_df.to_excel("filtered_test_data_v2.xlsx", index=False)
print("✅ Filtered test data saved as 'filtered_test_data_v2.xlsx'")


✅ Filtered test data saved as 'filtered_test_data_v2.xlsx'
